# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets available
record_sets = list(dataset.record_sets)
print("Record sets in the dataset (by @id):")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For each record set, print its fields
for rs in record_sets:
    print(f"\nFields for record set @id: {rs['@id']} ({rs.get('name','')})")
    fields = rs.get('field', [])
    for f in fields:
        # Each field is a dict with @id and optional name
        field_id = f['@id'] if isinstance(f, dict) else f
        # Try to display also the name if present in metadata
        field_obj = next((fld for fld in dataset.fields if fld['@id'] == field_id), None)
        field_name = field_obj['name'] if field_obj and 'name' in field_obj else 'N/A'
        print(f"  - Field @id: {field_id}, name: {field_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's select the main table with protein information. Suppose its @id is 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034#proteins'
# For demonstration, we'll dynamically take the first table-like recordset (should be adapted if multiple tables of interest)

main_record_set_id = None
for rs in dataset.record_sets:
    # Heuristically find a recordSet with 'protein' or just take the first
    name = rs.get('name', '').lower() if 'name' in rs else ''
    if 'protein' in name:
        main_record_set_id = rs['@id']
        break
if main_record_set_id is None and len(dataset.record_sets) > 0:
    main_record_set_id = dataset.record_sets[0]['@id']

print(f"Using record set @id: {main_record_set_id}")

# For further analysis, aggregate all record set @ids
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Load records from all record sets into DataFrames
dataframes = {}
for rsid in all_record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

if main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    print(f"Fields (@id) in main record set: {list(dataframes[main_record_set_id].columns)}")
    display(dataframes[main_record_set_id].head())
else:
    print("No data loaded for the selected record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Find a numeric field (e.g., coverage, MW, or a peptide count)
main_df = dataframes[main_record_set_id]

# Try heuristically to pick a numeric column by @id: look for 'coverage', 'MW', or 'count' in the id.
possible_numeric_cols = [col for col in main_df.columns if any(x in col.lower() for x in ['coverage', 'mw', 'count', 'abundance', 'peptide', 'intensity', 'score'])]
numeric_field = None
for col in possible_numeric_cols:
    # Try to cast to float; if >60% can be cast, assume numeric
    try:
        vals = pd.to_numeric(main_df[col], errors='coerce')
        ratio_numeric = np.sum(vals.notna()) / len(vals)
        if ratio_numeric > 0.6:
            numeric_field = col
            main_df[col] = vals
            print(f"Selected numeric field for EDA: {numeric_field}")
            break
    except:
        continue
if not numeric_field:
    print('No suitable numeric field found for EDA')

# Demonstrate basic filtering and normalization if a field was found
if numeric_field:
    threshold = main_df[numeric_field].mean()  # filter by above-average
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
    
    # Normalize the selected field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"First few normalized {numeric_field} values:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a category: look for string columns with <20 unique values
    group_field = None
    for col in main_df.columns:
        if col == numeric_field:
            continue
        if main_df[col].dtype == object and main_df[col].nunique() < 20:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    # If group_field exists, boxplot
    if group_field:
        plt.figure(figsize=(12,6))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR² mass spectrometry dataset using the Croissant schema and `mlcroissant`.
- The record sets and fields were explored programmatically by their `@id`, and data was extracted into DataFrames.
- Exploratory analysis identified key numeric variables, enabling filtering, normalization, and basic grouping operations.
- Visualizations showcased distributions and relationships in the data.

Further steps may include domain-specific statistical analysis or machine learning on selected protein features.